In [ ]:
!pip install -U transformers trl peft bitsandbytes datasets accelerate rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstall

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# WHY IS RELIABILITY EASIER TO PREDICT ON 2WIKI?

import os
import re
import json
import numpy as np
import pandas as pd

from datasets import load_dataset
from rank_bm25 import BM25Okapi

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


D = "/content/drive/MyDrive/hedge_run"
SEED = 42
K = 10



HP_FEAT = f"{D}/hotpot_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
HP_RES  = f"{D}/hotpot_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
HP_TEST = f"{D}/dpo_test_questions.json"

TW_FEAT = f"{D}/2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
TW_RES  = f"{D}/2wiki_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
TW_FULL = f"{D}/2wiki_full.json"
TW_TEST = f"{D}/2wiki_test_questions.json"


for p in [
    HP_FEAT, HP_RES, HP_TEST,
    TW_FEAT, TW_RES, TW_FULL, TW_TEST
]:
    assert os.path.exists(p), f"Missing: {p}"




REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]

CONFIDENCE = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]

FINAL = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]

ANSWER_LEVEL = CONFIDENCE + FINAL
FULL = REASONING + ANSWER_LEVEL



def tok(text):
    return re.findall(r"\w+", str(text).lower())


def norm_title(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())


def parse_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, np.integer)):
        return bool(x)

    s = str(x).strip().lower()

    if s in {"true", "1", "yes", "correct"}:
        return True
    if s in {"false", "0", "no", "wrong", "incorrect"}:
        return False

    raise ValueError(x)


def get_correct_col(df):

    for c in [
        "is_correct",
        "judge_correct",
        "correct",
    ]:
        if c in df.columns:
            return c

    raise KeyError(
        f"No correctness column. Columns:\n{list(df.columns)}"
    )


def get_qi_col(df):

    for c in [
        "question_index",
        "qi",
        "index",
    ]:
        if c in df.columns:
            return c

    raise KeyError(
        f"No question-index column. Columns:\n{list(df.columns)}"
    )


def load_test_ids(path):

    rows = json.load(open(path))

    return {
        int(
            r.get(
                "question_index",
                r.get("qi")
            )
        )
        for r in rows
    }


def load_dev_features(path, test_path):

    df = pd.read_csv(path)

    qi_col = get_qi_col(df)
    cor_col = get_correct_col(df)

    df["qi"] = df[qi_col].astype(int)
    df["correct"] = df[cor_col].map(parse_bool)

    test_ids = load_test_ids(test_path)

    dev = (
        df[
            ~df["qi"].isin(test_ids)
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(dev) == 800, len(dev)

    return dev


hp = load_dev_features(
    HP_FEAT,
    HP_TEST
)

tw = load_dev_features(
    TW_FEAT,
    TW_TEST
)


def oof_scores(df, features):

    X = (
        df[features]
        .astype(float)
        .values
    )

    # 1 = wrong
    y = (
        ~df["correct"]
    ).astype(int).values

    pred = np.zeros(len(df))

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    )

    for tr, va in cv.split(X, y):

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(
            X[tr]
        )

        Xva = scaler.transform(
            X[va]
        )

        clf = LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED,
        )

        clf.fit(
            Xtr,
            y[tr],
        )

        pred[va] = (
            clf.predict_proba(
                Xva
            )[:, 1]
        )

    return pred


for name, df in [
    ("HotpotQA", hp),
    ("2Wiki", tw),
]:

    df["risk_reasoning"] = oof_scores(
        df,
        REASONING
    )

    df["risk_answer"] = oof_scores(
        df,
        ANSWER_LEVEL
    )

    df["risk_full"] = oof_scores(
        df,
        FULL
    )

    y = (~df["correct"]).astype(int)

    print("\n" + "=" * 75)
    print(name, "— OOF SANITY CHECK")
    print("=" * 75)

    print(
        "Reasoning:",
        round(
            roc_auc_score(
                y,
                df["risk_reasoning"]
            ),
            4
        )
    )

    print(
        "Answer-level:",
        round(
            roc_auc_score(
                y,
                df["risk_answer"]
            ),
            4
        )
    )

    print(
        "Full:",
        round(
            roc_auc_score(
                y,
                df["risk_full"]
            ),
            4
        )
    )



hp_results = {
    int(r["question_index"]): r
    for r in json.load(open(HP_RES))
}

tw_results = {
    int(r["question_index"]): r
    for r in json.load(open(TW_RES))
}

tw_source = {
    int(r["question_index"]): r
    for r in json.load(open(TW_FULL))
}


print("\nLoading HotpotQA...")
hp_dataset = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation",
)

hp_by_question = {
    str(ex["question"]).strip(): ex
    for ex in hp_dataset
}


def context_items(ex):

    ctx = ex.get("context", {})

    titles = (
        ctx.get("title")
        or
        ctx.get("titles")
        or
        []
    )

    groups = ctx.get(
        "sentences",
        []
    )

    if groups and isinstance(
        groups[0],
        str
    ):
        groups = [groups]

    if isinstance(
        titles,
        str
    ):
        titles = [titles]

    titles = list(titles)

    if len(titles) < len(groups):
        titles += [
            f"passage {i}"
            for i in range(
                len(titles),
                len(groups)
            )
        ]

    items = []

    for title, sents in zip(
        titles,
        groups
    ):

        for sid, sent in enumerate(sents):

            sent = str(sent).strip()

            if sent:

                items.append({
                    "title":
                        norm_title(title),

                    "sid":
                        int(sid),

                    "text":
                        sent,
                })

    return items


def gold_pairs(ex):

    sf = (
        ex.get("supporting_facts")
        or
        ex.get("supporting_fact")
    )

    out = set()

    if sf is None:
        return out

    # Hotpot-style dict
    if isinstance(sf, dict):

        titles = (
            sf.get("title")
            or
            sf.get("titles")
            or
            []
        )

        ids = (
            sf.get("sent_id")
            or
            sf.get("sent_ids")
            or
            sf.get("sentence_id")
            or
            []
        )

        for title, sid in zip(
            titles,
            ids
        ):

            try:
                out.add(
                    (
                        norm_title(title),
                        int(sid)
                    )
                )
            except Exception:
                pass

        return out


    # 2Wiki can also appear as list
    if isinstance(sf, list):

        for item in sf:

            if (
                isinstance(item, (list, tuple))
                and len(item) >= 2
            ):

                try:
                    out.add(
                        (
                            norm_title(item[0]),
                            int(item[1])
                        )
                    )
                except Exception:
                    pass

            elif isinstance(item, dict):

                title = (
                    item.get("title")
                    or
                    item.get("doc")
                )

                sid = (
                    item.get("sent_id")
                    if "sent_id" in item
                    else item.get("sentence_id")
                )

                if (
                    title is not None
                    and
                    sid is not None
                ):
                    out.add(
                        (
                            norm_title(title),
                            int(sid)
                        )
                    )

    return out


def bm25_top(items, query, k=10):

    if not items:
        return []

    bm = BM25Okapi(
        [
            tok(x["text"])
            for x in items
        ]
    )

    scores = bm.get_scores(
        tok(query)
    )

    order = (
        np.argsort(scores)
        [::-1][:k]
    )

    return [
        items[int(i)]
        for i in order
    ]


def retrieve_recall(items, gold, query):

    if not gold:
        return np.nan, np.nan, np.nan

    retrieved = bm25_top(
        items,
        query,
        K,
    )

    got = {
        (x["title"], x["sid"])
        for x in retrieved
    }

    n = len(
        got & gold
    )

    recall = n / len(gold)

    return (
        recall,
        float(n > 0),
        float(n == len(gold))
    )


def extract_steps(record):

    steps = record.get(
        "selected_steps",
        []
    )

    out = []

    for x in steps:

        if isinstance(x, str):

            if x.strip():
                out.append(
                    x.strip()
                )

        elif isinstance(x, dict):

            text = (
                x.get("selected_text")
                or
                x.get("text")
                or
                x.get("step")
            )

            if text and str(text).strip():
                out.append(
                    str(text).strip()
                )

    return out


def lexical_overlap(
    question,
    items,
    gold
):

    q = set(
        tok(question)
    )

    if not q or not gold:
        return np.nan

    gold_text = []

    for x in items:

        if (
            x["title"],
            x["sid"]
        ) in gold:

            gold_text.extend(
                tok(x["text"])
            )

    g = set(gold_text)

    if not g:
        return np.nan

    # fraction of question terms
    # appearing in gold support
    return len(q & g) / len(q)




def analyse_example(
    qi,
    correct,
    result_record,
    source_ex,
    dataset_name,
):

    question = str(
        result_record.get(
            "question",
            source_ex.get(
                "question",
                ""
            )
        )
    ).strip()

    items = context_items(
        source_ex
    )

    gold = gold_pairs(
        source_ex
    )

    q_rec, q_any, q_all = (
        retrieve_recall(
            items,
            gold,
            question,
        )
    )

    steps = extract_steps(
        result_record
    )



    union_pairs = set()

    for step in steps:

        retrieved = bm25_top(
            items,
            question + " " + step,
            K,
        )

        union_pairs.update(
            (
                x["title"],
                x["sid"]
            )
            for x in retrieved
        )


    if gold and steps:

        traj_rec = (
            len(
                union_pairs & gold
            )
            /
            len(gold)
        )

        traj_all = float(
            len(
                union_pairs & gold
            )
            ==
            len(gold)
        )

    else:

        traj_rec = np.nan
        traj_all = np.nan


    qtype = None

    for key in [
        "type",
        "question_type",
        "q_type",
        "reasoning_type",
    ]:

        value = source_ex.get(
            key
        )

        if (
            value is not None
            and
            str(value).strip()
        ):

            qtype = str(
                value
            ).strip()

            break


    level = source_ex.get(
        "level"
    )


    return {
        "dataset":
            dataset_name,

        "qi":
            qi,

        "correct":
            bool(correct),

        "qtype":
            qtype,

        "level":
            level,

        "num_context_sentences":
            len(items),

        "num_gold_facts":
            len(gold),

        "num_steps":
            len(steps),

        "q_gold_recall10":
            q_rec,

        "q_any_gold10":
            q_any,

        "q_all_gold10":
            q_all,

        "traj_gold_recall":
            traj_rec,

        "traj_all_gold":
            traj_all,

        "question_gold_overlap":
            lexical_overlap(
                question,
                items,
                gold,
            ),
    }


rows = []


# HOTPOT
for _, r in hp.iterrows():

    qi = int(
        r["qi"]
    )

    result = hp_results[
        qi
    ]

    q = str(
        result["question"]
    ).strip()

    ex = hp_by_question.get(
        q
    )

    if ex is None:
        print(
            "Hotpot source missing:",
            qi
        )
        continue

    rows.append(
        analyse_example(
            qi,
            r["correct"],
            result,
            ex,
            "HotpotQA",
        )
    )


# 2WIKI
for _, r in tw.iterrows():

    qi = int(
        r["qi"]
    )

    result = tw_results[
        qi
    ]

    ex = tw_source[
        qi
    ]

    rows.append(
        analyse_example(
            qi,
            r["correct"],
            result,
            ex,
            "2Wiki",
        )
    )


diag = pd.DataFrame(
    rows
)


print(
    "\nDiagnostic records:",
    diag.groupby(
        "dataset"
    ).size().to_dict()
)




print("\n" + "=" * 80)
print("A. OVERALL EVIDENCE / RETRIEVAL")
print("=" * 80)

cols = [
    "num_context_sentences",
    "num_gold_facts",
    "q_gold_recall10",
    "q_all_gold10",
    "traj_gold_recall",
    "traj_all_gold",
    "question_gold_overlap",
]

print(
    diag.groupby(
        "dataset"
    )[cols]
    .mean()
    .round(4)
)




print("\n" + "=" * 80)
print("B. RETRIEVAL: CORRECT VS WRONG")
print("=" * 80)

print(
    diag.groupby(
        ["dataset", "correct"]
    )[
        [
            "q_gold_recall10",
            "q_all_gold10",
            "traj_gold_recall",
            "traj_all_gold",
            "question_gold_overlap",
        ]
    ]
    .agg(
        ["mean", "count"]
    )
    .round(4)
)



print("\n" + "=" * 80)
print("C. RETRIEVAL FAILURE -> WRONG ANSWER AUROC")
print("=" * 80)

for dataset in [
    "HotpotQA",
    "2Wiki",
]:

    d = diag[
        diag["dataset"] == dataset
    ].copy()

    y = (
        ~d["correct"]
    ).astype(int)

    print(
        f"\n{dataset}"
    )

    for col in [
        "q_gold_recall10",
        "traj_gold_recall",
        "question_gold_overlap",
    ]:

        mask = (
            d[col]
            .notna()
        )

        if (
            mask.sum() > 0
            and
            y[mask].nunique() == 2
        ):

            auc = roc_auc_score(
                y[mask],
                -d.loc[
                    mask,
                    col
                ],
            )

            print(
                f"  low {col:24s} -> wrong: "
                f"AUROC = {auc:.4f}"
            )



print("\n" + "=" * 80)
print("D. STRONGEST INDIVIDUAL RELIABILITY FEATURES")
print("=" * 80)

for dataset, df in [
    ("HotpotQA", hp),
    ("2Wiki", tw),
]:

    y = (
        ~df["correct"]
    ).astype(int)

    feature_rows = []

    for f in FULL:

        x = (
            df[f]
            .astype(float)
        )

        auc = roc_auc_score(
            y,
            x
        )

        discrim = max(
            auc,
            1 - auc
        )

        correct_mean = (
            x[df["correct"]]
            .mean()
        )

        wrong_mean = (
            x[~df["correct"]]
            .mean()
        )

        feature_rows.append({
            "feature":
                f,

            "AUC_directionless":
                discrim,

            "correct_mean":
                correct_mean,

            "wrong_mean":
                wrong_mean,
        })


    table = (
        pd.DataFrame(
            feature_rows
        )
        .sort_values(
            "AUC_directionless",
            ascending=False
        )
        .head(8)
    )

    print(
        f"\n{dataset}"
    )

    print(
        table.round(4)
        .to_string(
            index=False
        )
    )




risk_map = {}

for dataset, df in [
    ("HotpotQA", hp),
    ("2Wiki", tw),
]:

    for _, r in df.iterrows():

        risk_map[
            (
                dataset,
                int(r["qi"])
            )
        ] = {
            "reasoning":
                float(
                    r["risk_reasoning"]
                ),

            "full":
                float(
                    r["risk_full"]
                ),
        }


diag["reasoning_risk"] = [
    risk_map.get(
        (r.dataset, int(r.qi)),
        {}
    ).get(
        "reasoning",
        np.nan
    )
    for r in diag.itertuples()
]

diag["full_risk"] = [
    risk_map.get(
        (r.dataset, int(r.qi)),
        {}
    ).get(
        "full",
        np.nan
    )
    for r in diag.itertuples()
]


print("\n" + "=" * 80)
print("E. QUESTION-TYPE BREAKDOWN")
print("=" * 80)


for dataset in [
    "HotpotQA",
    "2Wiki",
]:

    d = diag[
        diag["dataset"] == dataset
    ].copy()

    print(
        f"\n{dataset} type values:"
    )

    print(
        d["qtype"]
        .value_counts(
            dropna=False
        )
    )


    type_rows = []

    for qtype, g in d.groupby(
        "qtype",
        dropna=False,
    ):

        row = {
            "type":
                qtype,

            "n":
                len(g),

            "accuracy":
                g["correct"].mean(),

            "q_recall10":
                g[
                    "q_gold_recall10"
                ].mean(),
        }


        y = (
            ~g["correct"]
        ).astype(int)

        if (
            len(g) >= 20
            and
            y.nunique() == 2
        ):

            row[
                "reasoning_AUROC"
            ] = roc_auc_score(
                y,
                g["reasoning_risk"]
            )

            row[
                "full_AUROC"
            ] = roc_auc_score(
                y,
                g["full_risk"]
            )

        else:

            row[
                "reasoning_AUROC"
            ] = np.nan

            row[
                "full_AUROC"
            ] = np.nan


        type_rows.append(
            row
        )


    print(
        pd.DataFrame(
            type_rows
        )
        .sort_values(
            "n",
            ascending=False
        )
        .round(4)
        .to_string(
            index=False
        )
    )


print("\nHotpotQA difficulty level:")

hp_levels = diag[
    diag["dataset"]
    ==
    "HotpotQA"
]

print(
    hp_levels.groupby(
        "level",
        dropna=False
    )
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        q_recall10=("q_gold_recall10", "mean"),
    )
    .round(4)
)



OUT = (
    f"{D}/rq1_cross_dataset_diagnostics.csv"
)

diag.to_csv(
    OUT,
    index=False
)

print(
    "\nSaved:",
    OUT
)


HotpotQA — OOF SANITY CHECK
Reasoning: 0.6103
Answer-level: 0.748
Full: 0.7511

2Wiki — OOF SANITY CHECK
Reasoning: 0.6978
Answer-level: 0.7806
Full: 0.8094

Loading HotpotQA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

...

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]


Diagnostic records: {'2Wiki': 800, 'HotpotQA': 800}

A. OVERALL EVIDENCE / RETRIEVAL
          num_context_sentences  num_gold_facts  q_gold_recall10  \
dataset                                                            
2Wiki                   33.1200           0.000              NaN   
HotpotQA                41.8412           2.405           0.7553   

          q_all_gold10  traj_gold_recall  traj_all_gold  question_gold_overlap  
dataset                                                                         
2Wiki              NaN               NaN            NaN                    NaN  
HotpotQA          0.52             0.864         0.7155                 0.6511  

B. RETRIEVAL: CORRECT VS WRONG
                 q_gold_recall10       q_all_gold10       traj_gold_recall  \
                            mean count         mean count             mean   
dataset  correct                                                             
2Wiki    False               NaN     0          NaN

In [ ]:
import json
from pprint import pprint

D = "/content/drive/MyDrive/hedge_run"

data = json.load(
    open(f"{D}/2wiki_full.json")
)

print("Top-level type:", type(data))
print("Number of records:", len(data))

if isinstance(data, dict):
    first_key = next(iter(data))
    ex = data[first_key]
    print("First key:", first_key)
else:
    ex = data[0]

print("\nTOP-LEVEL KEYS")
print(ex.keys())

print("\nFULL FIRST RECORD")
pprint(ex)

print("\nFIELDS THAT MAY CONTAIN TYPE / EVIDENCE")
for k, v in ex.items():
    kl = k.lower()
    if any(
        word in kl
        for word in [
            "support",
            "evidence",
            "fact",
            "type",
            "reason",
        ]
    ):
        print("\n", k, ":")
        pprint(v)

Top-level type: <class 'list'>
Number of records: 1000

TOP-LEVEL KEYS
dict_keys(['question_index', 'question', 'gold_answer', 'context'])

FULL FIRST RECORD
{'context': {'sentences': [['Drew Esocoff( born c. 1957) is an American '
                            'television sports director, who as of 2006 has '
                            'been the director of NBC Sunday Night Football.'],
                           ['The Fate of a Night( German: Das Schicksal einer '
                            'Nacht) is a 1927 German silent film directed by '
                            'Erich Schönfelder.',
                            "The film's art direction was by Ernst Stern."],
                           ['Ben Palmer is a British film and television '
                            'director who is known for being the director of" '
                            'Bo\' Selecta" and" The Inbetweeners".'],
                           ['Max and Helen is a 1990 American drama film '
                        

In [ ]:


import re
import json
import numpy as np
import pandas as pd

from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sklearn.metrics import roc_auc_score


D = "/content/drive/MyDrive/hedge_run"
K = 10




tw_feat = pd.read_csv(
    f"{D}/2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

tw_results = {
    int(r["question_index"]): r
    for r in json.load(
        open(
            f"{D}/2wiki_passive_iterative_v2_FINAL_CLEAN_LABELLED_1000.json"
        )
    )
}

test_rows = json.load(
    open(
        f"{D}/2wiki_test_questions.json"
    )
)

test_ids = {
    int(
        r.get(
            "question_index",
            r.get("qi")
        )
    )
    for r in test_rows
}


def find_col(df, names):
    for c in names:
        if c in df.columns:
            return c
    raise KeyError(names)


qi_col = find_col(
    tw_feat,
    [
        "question_index",
        "qi",
        "index",
    ]
)

correct_col = find_col(
    tw_feat,
    [
        "is_correct",
        "judge_correct",
        "correct",
    ]
)


def parse_bool(x):
    if isinstance(x, bool):
        return x

    if isinstance(x, (int, np.integer)):
        return bool(x)

    s = str(x).strip().lower()

    if s in {
        "true",
        "1",
        "correct",
        "yes",
    }:
        return True

    if s in {
        "false",
        "0",
        "incorrect",
        "wrong",
        "no",
    }:
        return False

    raise ValueError(x)


tw_feat["qi"] = (
    tw_feat[qi_col]
    .astype(int)
)

tw_feat["correct"] = (
    tw_feat[correct_col]
    .map(parse_bool)
)

tw_dev = (
    tw_feat[
        ~tw_feat["qi"].isin(
            test_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(tw_dev) == 800



print("Loading original 2Wiki development data...")

ds = load_dataset(
    "framolfese/2WikiMultihopQA",
    split="validation",
)

print(
    "Original dev examples:",
    len(ds)
)

print(
    "Fields:",
    ds.column_names
)


def norm_text(x):
    x = str(x).strip().lower()
    x = re.sub(
        r"\s+",
        " ",
        x
    )
    return x


def norm_title(x):
    return norm_text(x)


def tok(text):
    return re.findall(
        r"\w+",
        str(text).lower()
    )



original_by_question = {}

for ex in ds:

    q = norm_text(
        ex["question"]
    )

    original_by_question.setdefault(
        q,
        []
    ).append(ex)




matched = {}
not_found = []
ambiguous = []


for qi in tw_dev["qi"]:

    qi = int(qi)

    q = norm_text(
        tw_results[qi]["question"]
    )

    candidates = (
        original_by_question
        .get(
            q,
            []
        )
    )


    if len(candidates) == 1:

        matched[qi] = (
            candidates[0]
        )

    elif len(candidates) == 0:

        not_found.append(
            (
                qi,
                tw_results[qi]["question"]
            )
        )

    else:

        ambiguous.append(
            (
                qi,
                len(candidates),
                tw_results[qi]["question"]
            )
        )


print()
print("=" * 70)
print("MATCH CHECK")
print("=" * 70)

print(
    "Matched:",
    len(matched)
)

print(
    "Not found:",
    len(not_found)
)

print(
    "Ambiguous:",
    len(ambiguous)
)

if not_found:
    print(
        "\nFirst missing:",
        not_found[:5]
    )

if ambiguous:
    print(
        "\nFirst ambiguous:",
        ambiguous[:5]
    )


assert len(matched) == 800, (
    "Do not continue until all 800 "
    "development examples are matched."
)



def context_items(ex):

    ctx = ex["context"]

    titles = (
        ctx["title"]
    )

    sentence_groups = (
        ctx["sentences"]
    )

    items = []

    for title, sents in zip(
        titles,
        sentence_groups
    ):

        for sid, sent in enumerate(
            sents
        ):

            sent = str(
                sent
            ).strip()

            if sent:

                items.append({
                    "title":
                        norm_title(title),

                    "sid":
                        int(sid),

                    "text":
                        sent,
                })

    return items


def supporting_pairs(ex):

    sf = ex[
        "supporting_facts"
    ]

    titles = (
        sf["title"]
    )

    ids = (
        sf["sent_id"]
    )

    return {
        (
            norm_title(title),
            int(sid)
        )
        for title, sid
        in zip(
            titles,
            ids
        )
    }




def bm25_top(
    items,
    query,
    k=10
):

    bm = BM25Okapi(
        [
            tok(x["text"])
            for x in items
        ]
    )

    scores = (
        bm.get_scores(
            tok(query)
        )
    )

    order = (
        np.argsort(scores)
        [::-1][:k]
    )

    return [
        items[int(i)]
        for i in order
    ]


def recall_at_10(
    items,
    gold,
    query
):

    retrieved = bm25_top(
        items,
        query,
        K
    )

    got = {
        (
            x["title"],
            x["sid"]
        )
        for x in retrieved
    }

    hit = len(
        got & gold
    )

    return {
        "recall":
            hit / len(gold),

        "any":
            float(hit > 0),

        "all":
            float(
                hit == len(gold)
            ),
    }




def extract_steps(record):

    steps = record.get(
        "selected_steps",
        []
    )

    out = []

    for x in steps:

        if isinstance(
            x,
            str
        ):

            if x.strip():
                out.append(
                    x.strip()
                )

        elif isinstance(
            x,
            dict
        ):

            text = (
                x.get(
                    "selected_text"
                )
                or
                x.get("text")
                or
                x.get("step")
            )

            if (
                text is not None
                and
                str(text).strip()
            ):

                out.append(
                    str(text).strip()
                )

    return out




rows = []


for _, row in tw_dev.iterrows():

    qi = int(
        row["qi"]
    )

    correct = bool(
        row["correct"]
    )

    result = tw_results[
        qi
    ]

    ex = matched[
        qi
    ]

    question = str(
        result["question"]
    ).strip()

    items = context_items(
        ex
    )

    gold = supporting_pairs(
        ex
    )




    q_ret = recall_at_10(
        items,
        gold,
        question
    )



    steps = extract_steps(
        result
    )

    union = set()

    for step in steps:

        retrieved = bm25_top(
            items,
            question + " " + step,
            K,
        )

        union.update(
            (
                x["title"],
                x["sid"]
            )
            for x in retrieved
        )


    if steps:

        hit = len(
            union & gold
        )

        traj_recall = (
            hit / len(gold)
        )

        traj_all = float(
            hit == len(gold)
        )

    else:

        traj_recall = np.nan
        traj_all = np.nan


    rows.append({

        "qi":
            qi,

        "correct":
            correct,

        "qtype":
            ex.get(
                "type",
                None
            ),

        "num_gold_facts":
            len(gold),

        "num_context_sentences":
            len(items),

        "q_gold_recall10":
            q_ret["recall"],

        "q_any_gold10":
            q_ret["any"],

        "q_all_gold10":
            q_ret["all"],

        "traj_gold_recall":
            traj_recall,

        "traj_all_gold":
            traj_all,
    })


diag2 = pd.DataFrame(
    rows
)


print()
print("=" * 80)
print("A. 2WIKI OVERALL RETRIEVAL")
print("=" * 80)

print(
    diag2[
        [
            "num_context_sentences",
            "num_gold_facts",
            "q_gold_recall10",
            "q_all_gold10",
            "traj_gold_recall",
            "traj_all_gold",
        ]
    ]
    .mean()
    .round(4)
)




print()
print("=" * 80)
print("B. 2WIKI RETRIEVAL: CORRECT VS WRONG")
print("=" * 80)

print(
    diag2.groupby(
        "correct"
    )[
        [
            "q_gold_recall10",
            "q_all_gold10",
            "traj_gold_recall",
            "traj_all_gold",
        ]
    ]
    .agg(
        ["mean", "count"]
    )
    .round(4)
)



print()
print("=" * 80)
print("C. RETRIEVAL FAILURE -> WRONG")
print("=" * 80)

y = (
    ~diag2[
        "correct"
    ]
).astype(int)


for col in [
    "q_gold_recall10",
    "traj_gold_recall",
]:

    mask = (
        diag2[col]
        .notna()
    )

    auc = roc_auc_score(
        y[mask],
        -diag2.loc[
            mask,
            col
        ],
    )

    print(
        f"low {col:22s} "
        f"-> wrong AUROC = "
        f"{auc:.4f}"
    )



print()
print("=" * 80)
print("D. 2WIKI QUESTION TYPES")
print("=" * 80)

print(
    diag2[
        "qtype"
    ]
    .value_counts(
        dropna=False
    )
)


print()

print(
    diag2.groupby(
        "qtype"
    )
    .agg(
        n=(
            "correct",
            "size"
        ),

        accuracy=(
            "correct",
            "mean"
        ),

        q_recall10=(
            "q_gold_recall10",
            "mean"
        ),

        traj_recall=(
            "traj_gold_recall",
            "mean"
        ),
    )
    .round(4)
    .sort_values(
        "n",
        ascending=False
    )
)



OUT = (
    f"{D}/rq1_2wiki_gold_retrieval_diagnostics.csv"
)

diag2.to_csv(
    OUT,
    index=False
)

print(
    "\nSaved:",
    OUT
)

Loading original 2Wiki development data...


README.md:   0%|          | 0.00/5.46k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  165MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/167454 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12576 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12576 [00:00<?, ? examples/s]

Original dev examples: 12576
Fields: ['id', 'question', 'answer', 'type', 'evidences', 'supporting_facts', 'context']

MATCH CHECK
Matched: 800
Not found: 0
Ambiguous: 0

A. 2WIKI OVERALL RETRIEVAL
num_context_sentences    33.1200
num_gold_facts            2.3963
q_gold_recall10           0.5686
q_all_gold10              0.2950
traj_gold_recall          0.7506
traj_all_gold             0.5726
dtype: float64

B. 2WIKI RETRIEVAL: CORRECT VS WRONG
        q_gold_recall10       q_all_gold10       traj_gold_recall        \
                   mean count         mean count             mean count   
correct                                                                   
False            0.4526   484       0.1446   484           0.6620   442   
True             0.7463   316       0.5253   316           0.8745   316   

        traj_all_gold        
                 mean count  
correct                      
False          0.4525   442  
True           0.7405   316  

C. RETRIEVAL FAILURE -> 

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

D = "/content/drive/MyDrive/hedge_run"
SEED = 42

REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]



feat = pd.read_csv(
    f"{D}/2wiki_passive_iterative_v2_FINAL_CLEAN_features17_LABELLED.csv"
)

diag = pd.read_csv(
    f"{D}/rq1_2wiki_gold_retrieval_diagnostics.csv"
)

test = json.load(
    open(f"{D}/2wiki_test_questions.json")
)

test_ids = {
    int(r.get("question_index", r.get("qi")))
    for r in test
}

# identify columns
qi_col = (
    "question_index"
    if "question_index" in feat.columns
    else "qi"
)

correct_col = next(
    c for c in [
        "is_correct",
        "judge_correct",
        "correct"
    ]
    if c in feat.columns
)

feat["qi"] = feat[qi_col].astype(int)

def parse_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).lower() in {
        "true", "1", "correct", "yes"
    }

feat["correct"] = feat[correct_col].map(parse_bool)

feat = feat[
    ~feat["qi"].isin(test_ids)
].copy()

assert len(feat) == 800

diag["qi"] = diag["qi"].astype(int)

df = feat.merge(
    diag[["qi", "qtype"]],
    on="qi",
    how="inner",
)

assert len(df) == 800



print("=" * 80)
print("A. REASONING PATTERNS BY QUESTION TYPE")
print("=" * 80)

summary = (
    df.groupby("qtype")
    .agg(
        n=("correct", "size"),
        accuracy=("correct", "mean"),
        num_steps=("num_steps", "mean"),
        mean_support=("mean_support", "mean"),
        min_support=("min_support_score", "mean"),
        conflict=("conflict", "mean"),
        support_spread=("support_spread", "mean"),
        max_contradiction=("max_contradiction_score", "mean"),
    )
    .round(4)
)

print(summary)




def oof_classifier(data, features):

    X = data[features].astype(float).values
    y = (~data["correct"]).astype(int).values

    pred = np.zeros(len(data))

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    )

    for tr, va in cv.split(X, y):

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(X[tr])
        Xva = scaler.transform(X[va])

        clf = LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED,
        )

        clf.fit(Xtr, y[tr])

        pred[va] = clf.predict_proba(Xva)[:, 1]

    return pred


df["global_reasoning_risk"] = oof_classifier(
    df,
    REASONING,
)

y = (~df["correct"]).astype(int)

print("\n" + "=" * 80)
print("B. GLOBAL REASONING AUROC")
print("=" * 80)

print(
    "Overall:",
    round(
        roc_auc_score(
            y,
            df["global_reasoning_risk"],
        ),
        4,
    ),
)




print("\n" + "=" * 80)
print("C. REASONING AUROC WITHIN QUESTION TYPES")
print("=" * 80)

type_aucs = []

for qtype, g in df.groupby("qtype"):

    yt = (~g["correct"]).astype(int)

    auc = roc_auc_score(
        yt,
        g["global_reasoning_risk"],
    )

    nc = int(g["correct"].sum())
    nw = len(g) - nc

    type_aucs.append({
        "qtype": qtype,
        "n": len(g),
        "correct": nc,
        "wrong": nw,
        "AUROC": auc,
        "pair_weight": nc * nw,
    })

type_aucs = pd.DataFrame(type_aucs)

print(
    type_aucs[
        ["qtype", "n", "correct", "wrong", "AUROC"]
    ]
    .round(4)
    .to_string(index=False)
)


# Pair-weighted within-type AUROC
within_auc = np.average(
    type_aucs["AUROC"],
    weights=type_aucs["pair_weight"],
)

print(
    "\nPair-weighted within-type AUROC:",
    round(within_auc, 4)
)



print("\n" + "=" * 80)
print("D. QUESTION-TYPE-ONLY AUROC")
print("=" * 80)

y = (~df["correct"]).astype(int).values

type_pred = np.zeros(len(df))

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

for tr, va in cv.split(df, y):

    train = df.iloc[tr].copy()
    valid = df.iloc[va].copy()

    train["wrong"] = (
        ~train["correct"]
    ).astype(int)

    type_risk = (
        train.groupby("qtype")["wrong"]
        .mean()
        .to_dict()
    )

    fallback = train["wrong"].mean()

    type_pred[va] = (
        valid["qtype"]
        .map(type_risk)
        .fillna(fallback)
        .values
    )

type_auc = roc_auc_score(
    y,
    type_pred,
)

print(
    "Question-type-only AUROC:",
    round(type_auc, 4)
)



print("\n" + "=" * 80)
print("E. SEPARATE WITHIN-TYPE REASONING CLASSIFIERS")
print("=" * 80)

rows = []

for qtype, g in df.groupby("qtype"):

    g = g.reset_index(drop=True)

    y = (~g["correct"]).astype(int)

    pred = oof_classifier(
        g,
        REASONING,
    )

    auc = roc_auc_score(
        y,
        pred,
    )

    rows.append({
        "qtype": qtype,
        "n": len(g),
        "accuracy": g["correct"].mean(),
        "reasoning_AUROC": auc,
    })

within_models = pd.DataFrame(rows)

print(
    within_models
    .round(4)
    .to_string(index=False)
)


print("\n" + "=" * 80)
print("KEY COMPARISON")
print("=" * 80)

print(
    "Overall reasoning AUROC:        ",
    round(
        roc_auc_score(
            (~df["correct"]).astype(int),
            df["global_reasoning_risk"],
        ),
        4,
    )
)

print(
    "Question-type-only AUROC:       ",
    round(type_auc, 4)
)

print(
    "Pair-weighted within-type AUROC:",
    round(within_auc, 4)
)

A. REASONING PATTERNS BY QUESTION TYPE
                     n  accuracy  num_steps  mean_support  min_support  \
qtype                                                                    
bridge_comparison  157    0.6561     3.5350        0.7209       0.3456   
comparison         189    0.7090     2.5714        0.8106       0.5713   
compositional      353    0.1728     1.6204        0.7257       0.6160   
inference          101    0.1782     1.7426        0.4482       0.3011   

                   conflict  support_spread  max_contradiction  
qtype                                                           
bridge_comparison    0.8917          0.6031             0.9258  
comparison           0.8839          0.3992             0.9068  
compositional        0.5932          0.2070             0.6596  
inference            0.4870          0.2793             0.7492  

B. GLOBAL REASONING AUROC
Overall: 0.6978

C. REASONING AUROC WITHIN QUESTION TYPES
            qtype   n  correct  wrong  AU

In [ ]:


import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression



D = "/content/drive/MyDrive/hedge_run"
SEED = 42
N_BOOT = 5000


FEATURE_FILE = (
    f"{D}/2wiki_passive_iterative_v2_"
    "FINAL_CLEAN_features17_LABELLED.csv"
)

TYPE_FILE = (
    f"{D}/rq1_2wiki_gold_retrieval_diagnostics.csv"
)

TEST_FILE = (
    f"{D}/2wiki_test_questions.json"
)



REASONING = [
    "max_contradiction_score",
    "any_contradicted",
    "min_support_score",
    "num_steps",
    "frac_supported",
    "frac_contradicted",
    "frac_unclear",
    "mean_support",
    "mean_contradiction",
    "conflict",
    "support_spread",
]

CONFIDENCE = [
    "self_consistency",
    "answer_norm_entropy",
    "answer_logprob",
]

FINAL_VERIFY = [
    "fa_support_retr",
    "fa_contradiction_retr",
    "fa_contradicted_retr",
]

ANSWER_LEVEL = (
    CONFIDENCE
    +
    FINAL_VERIFY
)

FULL = (
    REASONING
    +
    ANSWER_LEVEL
)



def parse_bool(x):

    if isinstance(x, bool):
        return x

    if isinstance(x, (int, np.integer)):
        return bool(x)

    s = str(x).strip().lower()

    if s in {
        "true",
        "1",
        "yes",
        "correct",
    }:
        return True

    if s in {
        "false",
        "0",
        "no",
        "wrong",
        "incorrect",
    }:
        return False

    raise ValueError(
        f"Cannot parse boolean: {x}"
    )


def find_col(df, options):

    for c in options:

        if c in df.columns:
            return c

    raise KeyError(
        f"None of {options} found.\n"
        f"Columns: {list(df.columns)}"
    )




feat = pd.read_csv(
    FEATURE_FILE
)

types = pd.read_csv(
    TYPE_FILE
)

test_rows = json.load(
    open(TEST_FILE)
)


test_ids = {

    int(
        r.get(
            "question_index",
            r.get("qi")
        )
    )

    for r in test_rows
}


qi_col = find_col(
    feat,
    [
        "question_index",
        "qi",
        "index",
    ],
)

correct_col = find_col(
    feat,
    [
        "is_correct",
        "judge_correct",
        "correct",
    ],
)


feat["qi"] = (
    feat[qi_col]
    .astype(int)
)

feat["correct"] = (
    feat[correct_col]
    .map(parse_bool)
)


feat = (
    feat[
        ~feat["qi"].isin(
            test_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(feat) == 800


types["qi"] = (
    types["qi"]
    .astype(int)
)


df = feat.merge(
    types[
        [
            "qi",
            "qtype",
        ]
    ],
    on="qi",
    how="inner",
)

assert len(df) == 800
assert df["qtype"].notna().all()


print(
    "Question types:"
)

print(
    df["qtype"]
    .value_counts()
)

print(
    "\nOverall accuracy:",
    round(
        df["correct"].mean(),
        4
    )
)



type_dummies = pd.get_dummies(
    df["qtype"],
    prefix="type",
    dtype=float,
)

df = pd.concat(
    [
        df,
        type_dummies,
    ],
    axis=1,
)

TYPE_FEATURES = (
    type_dummies
    .columns
    .tolist()
)

print(
    "\nType features:",
    TYPE_FEATURES
)



def oof_logistic(
    data,
    feature_cols,
    n_splits=5,
):

    X = (
        data[feature_cols]
        .astype(float)
        .values
    )

    # 1 = wrong
    y = (
        ~data["correct"]
    ).astype(int).values

    pred = np.zeros(
        len(data),
        dtype=float,
    )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED,
    )

    for tr, va in cv.split(
        X,
        y,
    ):

        scaler = StandardScaler()

        Xtr = scaler.fit_transform(
            X[tr]
        )

        Xva = scaler.transform(
            X[va]
        )

        clf = LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=SEED,
        )

        clf.fit(
            Xtr,
            y[tr],
        )

        pred[va] = (
            clf.predict_proba(
                Xva
            )[:, 1]
        )

    return pred



MODELS = {

    "Answer-level":
        ANSWER_LEVEL,

    "Question type":
        TYPE_FEATURES,

    "Answer-level + type":
        ANSWER_LEVEL
        +
        TYPE_FEATURES,

    "Full 17":
        FULL,

    "Full 17 + type":
        FULL
        +
        TYPE_FEATURES,
}


predictions = {}

y = (
    ~df["correct"]
).astype(int).values


print()
print("=" * 80)
print("A. MAIN COMPARISON")
print("=" * 80)


main_rows = []


for name, features in MODELS.items():

    pred = oof_logistic(
        df,
        features,
    )

    predictions[name] = pred

    auc = roc_auc_score(
        y,
        pred,
    )

    main_rows.append({
        "Model":
            name,

        "Features":
            len(features),

        "AUROC":
            auc,
    })


main_table = pd.DataFrame(
    main_rows
)

print(
    main_table
    .round(4)
    .to_string(
        index=False
    )
)




def paired_bootstrap_auc(
    y,
    pred_a,
    pred_b,
    n_boot=N_BOOT,
    seed=SEED,
):

    """
    Returns delta:
        AUROC(B) - AUROC(A)
    """

    rng = np.random.default_rng(
        seed
    )

    n = len(y)

    observed = (
        roc_auc_score(
            y,
            pred_b,
        )
        -
        roc_auc_score(
            y,
            pred_a,
        )
    )

    deltas = []


    for _ in range(n_boot):

        idx = rng.integers(
            0,
            n,
            n,
        )

        yb = y[idx]

        # skip samples containing only one class
        if len(
            np.unique(yb)
        ) < 2:

            continue

        auc_a = roc_auc_score(
            yb,
            pred_a[idx],
        )

        auc_b = roc_auc_score(
            yb,
            pred_b[idx],
        )

        deltas.append(
            auc_b
            -
            auc_a
        )


    deltas = np.asarray(
        deltas
    )


    lo, hi = np.percentile(
        deltas,
        [
            2.5,
            97.5,
        ]
    )


    p_le_zero = (
        np.mean(
            deltas <= 0
        )
    )


    p_two = min(
        1.0,
        2 * min(
            p_le_zero,
            np.mean(
                deltas >= 0
            ),
        ),
    )


    return {
        "delta":
            observed,

        "CI_low":
            lo,

        "CI_high":
            hi,

        "P_delta_le_0":
            p_le_zero,

        "approx_p_two_sided":
            p_two,
    }




print()
print("=" * 80)
print("B. PAIRED BOOTSTRAP COMPARISONS")
print("=" * 80)


comparisons = [

    (
        "Answer-level",
        "Full 17",
    ),

    (
        "Answer-level",
        "Answer-level + type",
    ),

    (
        "Answer-level + type",
        "Full 17",
    ),

    (
        "Full 17",
        "Full 17 + type",
    ),

    (
        "Answer-level + type",
        "Full 17 + type",
    ),
]


bootstrap_rows = []


for a, b in comparisons:

    result = paired_bootstrap_auc(
        y,
        predictions[a],
        predictions[b],
    )

    bootstrap_rows.append({

        "Comparison":
            f"{a} -> {b}",

        **result,
    })


boot_table = pd.DataFrame(
    bootstrap_rows
)

print(
    boot_table
    .round(4)
    .to_string(
        index=False
    )
)


print()
print("=" * 80)
print("C. GLOBAL MODELS EVALUATED WITHIN EACH QUESTION TYPE")
print("=" * 80)


df[
    "risk_answer"
] = predictions[
    "Answer-level"
]

df[
    "risk_full"
] = predictions[
    "Full 17"
]


within_global_rows = []


for qtype, g in df.groupby(
    "qtype"
):

    yt = (
        ~g["correct"]
    ).astype(int)

    auc_answer = roc_auc_score(
        yt,
        g["risk_answer"],
    )

    auc_full = roc_auc_score(
        yt,
        g["risk_full"],
    )

    nc = int(
        g["correct"]
        .sum()
    )

    nw = (
        len(g)
        -
        nc
    )


    within_global_rows.append({

        "qtype":
            qtype,

        "n":
            len(g),

        "accuracy":
            g[
                "correct"
            ].mean(),

        "Answer_AUROC":
            auc_answer,

        "Full_AUROC":
            auc_full,

        "Delta":
            auc_full
            -
            auc_answer,

        "pair_weight":
            nc * nw,
    })


within_global = pd.DataFrame(
    within_global_rows
)


print(
    within_global[
        [
            "qtype",
            "n",
            "accuracy",
            "Answer_AUROC",
            "Full_AUROC",
            "Delta",
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)


# Pair-weighted summary
weighted_answer = np.average(
    within_global[
        "Answer_AUROC"
    ],
    weights=within_global[
        "pair_weight"
    ],
)

weighted_full = np.average(
    within_global[
        "Full_AUROC"
    ],
    weights=within_global[
        "pair_weight"
    ],
)


print()

print(
    "Pair-weighted within-type "
    "Answer-level AUROC:",
    round(
        weighted_answer,
        4
    )
)

print(
    "Pair-weighted within-type "
    "Full AUROC:",
    round(
        weighted_full,
        4
    )
)

print(
    "Pair-weighted delta:",
    round(
        weighted_full
        -
        weighted_answer,
        4
    )
)




print()
print("=" * 80)
print("D. SEPARATE WITHIN-TYPE CLASSIFIERS")
print("=" * 80)


within_model_rows = []


for qtype, g in df.groupby(
    "qtype"
):

    g = (
        g.copy()
        .reset_index(
            drop=True
        )
    )

    yt = (
        ~g["correct"]
    ).astype(int).values


    answer_pred = oof_logistic(
        g,
        ANSWER_LEVEL,
        n_splits=5,
    )

    full_pred = oof_logistic(
        g,
        FULL,
        n_splits=5,
    )


    auc_answer = roc_auc_score(
        yt,
        answer_pred,
    )

    auc_full = roc_auc_score(
        yt,
        full_pred,
    )


    within_model_rows.append({

        "qtype":
            qtype,

        "n":
            len(g),

        "accuracy":
            g[
                "correct"
            ].mean(),

        "Answer_AUROC":
            auc_answer,

        "Full_AUROC":
            auc_full,

        "Delta":
            auc_full
            -
            auc_answer,
    })


within_models = pd.DataFrame(
    within_model_rows
)


print(
    within_models
    .round(4)
    .to_string(
        index=False
    )
)




def get_auc(name):

    return roc_auc_score(
        y,
        predictions[name],
    )


print()
print("=" * 80)
print("FINAL KEY NUMBERS")
print("=" * 80)

print(
    "Answer-level:          ",
    round(
        get_auc(
            "Answer-level"
        ),
        4
    )
)

print(
    "Question type only:    ",
    round(
        get_auc(
            "Question type"
        ),
        4
    )
)

print(
    "Answer-level + type:   ",
    round(
        get_auc(
            "Answer-level + type"
        ),
        4
    )
)

print(
    "Full 17:               ",
    round(
        get_auc(
            "Full 17"
        ),
        4
    )
)

print(
    "Full 17 + type:        ",
    round(
        get_auc(
            "Full 17 + type"
        ),
        4
    )
)

print()
print(
    "Original reasoning gain:",
    round(
        get_auc(
            "Full 17"
        )
        -
        get_auc(
            "Answer-level"
        ),
        4
    )
)

print(
    "Type gain over answer: ",
    round(
        get_auc(
            "Answer-level + type"
        )
        -
        get_auc(
            "Answer-level"
        ),
        4
    )
)

print(
    "Reasoning gain after "
    "type is available:    ",
    round(
        get_auc(
            "Full 17 + type"
        )
        -
        get_auc(
            "Answer-level + type"
        ),
        4
    )
)



OUT_MAIN = (
    f"{D}/rq1_2wiki_question_type_main_models.csv"
)

OUT_WITHIN = (
    f"{D}/rq1_2wiki_question_type_within_models.csv"
)

main_table.to_csv(
    OUT_MAIN,
    index=False,
)

within_models.to_csv(
    OUT_WITHIN,
    index=False,
)

print()
print(
    "Saved:",
    OUT_MAIN
)

print(
    "Saved:",
    OUT_WITHIN
)

Question types:
qtype
compositional        353
comparison           189
bridge_comparison    157
inference            101
Name: count, dtype: int64

Overall accuracy: 0.395

Type features: ['type_bridge_comparison', 'type_comparison', 'type_compositional', 'type_inference']

A. MAIN COMPARISON
              Model  Features  AUROC
       Answer-level         6 0.7806
      Question type         4 0.7321
Answer-level + type        10 0.8288
            Full 17        17 0.8094
     Full 17 + type        21 0.8313

B. PAIRED BOOTSTRAP COMPARISONS
                           Comparison   delta  CI_low  CI_high  P_delta_le_0  approx_p_two_sided
              Answer-level -> Full 17  0.0288  0.0071   0.0503        0.0042              0.0084
  Answer-level -> Answer-level + type  0.0482  0.0274   0.0696        0.0000              0.0000
       Answer-level + type -> Full 17 -0.0194 -0.0399   0.0013        0.9678              0.0644
            Full 17 -> Full 17 + type  0.0219  0.0072   0.0362

In [ ]:
import json

D = "/content/drive/MyDrive/hedge_run"

VERIFIER_JSON = f"{D}/verifier_full_claim_FINAL_TEST_judged.json"
FAIR_BASE     = f"{D}/base_point_fair.json"

verifier = json.load(open(VERIFIER_JSON))
fair = json.load(open(FAIR_BASE))


base_correct = {
    int(k): bool(v)
    for k, v in fair["per_q"].items()
}

assert len(base_correct) == 200
assert sum(base_correct.values()) == 96

records = {
    int(r["question_index"]): r
    for r in verifier["records"]
}

def state(r):
    if r["kind"] == "commit_correct":
        return "correct"
    if r["kind"] == "commit_wrong":
        return "wrong"
    if r["kind"] == "abstain":
        return "abstain"
    raise ValueError(r["kind"])

states = {
    qi: state(r)
    for qi, r in records.items()
    if qi in base_correct
}

assert len(states) == 200

correct = sum(s == "correct" for s in states.values())
wrong   = sum(s == "wrong" for s in states.values())
abstain = sum(s == "abstain" for s in states.values())

# question-level over-abstention
over_count = sum(
    states[qi] == "abstain" and base_correct[qi]
    for qi in states
)

over_abstention = over_count / 96

# abstention on baseline-wrong cases
wrong_base_count = 104

abstain_wrong_base = sum(
    states[qi] == "abstain" and not base_correct[qi]
    for qi in states
)

abstention_recall = abstain_wrong_base / wrong_base_count

# final common-baseline THS
N = 200
Pc0 = 96 / N
Pw0 = 104 / N
Pc  = correct / N
Pw  = wrong / N

ths = ((Pc * Pw0) - (Pw * Pc0)) / Pw0 * 100

print("FINAL HOTPOT VERIFIER — CLEAN COMMON BASELINE")
print("baseline correct:", 96)
print("baseline wrong:", 104)
print()
print("correct committed:", correct)
print("wrong committed:", wrong)
print("abstained:", abstain)
print()
print("coverage:", (correct + wrong) / N)
print("selective accuracy:", correct / (correct + wrong))
print("confident error:", wrong / N)
print("over-abstention count:", over_count)
print("over-abstention:", over_abstention)
print("abstention recall:", abstention_recall)
print("THS x100:", ths)

FINAL HOTPOT VERIFIER — CLEAN COMMON BASELINE
baseline correct: 96
baseline wrong: 104

correct committed: 108
wrong committed: 34
abstained: 58

coverage: 0.71
selective accuracy: 0.7605633802816901
confident error: 0.17
over-abstention count: 28
over-abstention: 0.2916666666666667
abstention recall: 0.28846153846153844
THS x100: 38.307692307692314
